# Week 4: Malicious Software

This notebook goes over key behavious related to malware.

- file integrity checking
- signature scanning
- worm propagation

# Section 1: File Integrity Checker

Key Concepts:

- Integrity Checking: A preventive technique that detects unauthorized changes to files or system components.
- Hashing (SHA-256): Generates a unique fingerprint for each file; even a one-bit change produces a completely different hash.
- Baseline Comparison: Security systems record “clean” file hashes and later compare them to detect tampering or infection.
- Malware Connection: Many viruses modify executable files — this method detects such unauthorized changes.

The code below is a script which loops through the files in the folder ```files``` and generates a SHA-256 hash for each. The results are saved in a CSV file with the file name, hash and time stamp.

In [7]:
import os, hashlib, csv
from datetime import datetime

folder_path = './files'

records = []
csv_output = "file_hashes.csv"

# loop through files in the folder
for filename in os.listdir(folder_path):
    file_path = os.path.join(folder_path, filename)

    if os.path.isfile(file_path):
        # compute hash
        hash = hashlib.sha256()
        with open(file_path, "rb") as f:
            # hash file in chunks of 4096 bytes
            for block in iter(lambda: f.read(4096), b""):
                hash.update(block)
        
        hash_hex = hash.hexdigest()
        timestamp = datetime.now().isoformat()

        records.append((filename, hash_hex, timestamp))

# create CSV file and write records
with open(csv_output, "w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["Filename", "SHA256 Hash", "Timestamp"])
    writer.writerows(records)

print(f"Hashes written to {csv_output}")

Hashes written to file_hashes.csv


Security systems rely on file hashes instead of time stamps or file sizes because timestamps and file sizes are easily manipulated. A malicious actor can change a file without affecting the timestamp or size in a noticeable way. Hashes (such as SHA-256) provide a unique fingerprint of a file's content. Even a single-byte change will result in a completely different hash, making it obvious when someone has tampered with a file. This is a strong method of integrity verification.

Malware can try to replace legitimate files with malicious ones of the same size to fool naive checks based on size alone. Sophisticated malware can modify a file and restore the original hash by exploiting weak hashing algorithms or using a hash collision attack. A collision attack is weakness where two different inputs can produce the exact same, identical hash values. This is rare, but it can happen. Lastly, some malware will entirely disable or bypass integrity-checking software.

What happens if a legitimate system update changes many hashes?
When an update legitimately modifies files, their hashes will naturally change. If the system is too strict, it would flag these as false-positives. To avoid this a system should:
1. Maintain a whitelist of trusted software updates. 
2. Use signed update packages to verify the authenticty of an update before updating the hash database.
3. Updating the hashes immediately after an update, so that they match the most up to date files.

# Section 2: Detecting suspicious file changes

This section simulates how a malware detection software would intentify file tampering by comparing current file hashes to a trusted bassline.

- Change Detection: Compares new hashes against the baseline to flag altered or deleted files.
- Heuristic Insight: Sudden modification of system or executable files often indicates infection.
- Forensics Use: Detecting when and which files changed helps trace intrusion paths.

The code below takes the files in the folder and compares them to the previously created CSV.

In [8]:
import os, hashlib, csv
from datetime import datetime

folder_path = './files'

new_hashes = []
csv_output = "file_hashes.csv"

# loop through files in the folder
for filename in os.listdir(folder_path):
    file_path = os.path.join(folder_path, filename)

    if os.path.isfile(file_path):
        # compute hash
        hash = hashlib.sha256()
        with open(file_path, "rb") as f:
            # hash file in chunks of 4096 bytes
            for block in iter(lambda: f.read(4096), b""):
                hash.update(block)
        
        hash_hex = hash.hexdigest()
        timestamp = datetime.now().isoformat()

        new_hashes.append((filename, hash_hex, timestamp))


# load previous hashes from CSV into a dictionary

previous_hashes = {}

try:
    with open(csv_output, "r", newline="") as csvf:
        reader = csv.DictReader(csvf)
        for row in reader:
            previous_hashes[row["Filename"]] = row["SHA256 Hash"]
except FileNotFoundError:
    print("File not found.")

# compare new hashes with previous hashes
mismatches = []
for new_filename, new_hash, new_timestamp in new_hashes:
    if new_filename in previous_hashes:
        # file exists in baseline - check if hash matches
        if previous_hashes[new_filename] != new_hash:
            mismatches.append({
                "filename": new_filename,
                "status": "MODIFIED",
                "old_hash": previous_hashes[new_filename],
                "new_hash": new_hash,
                "timestamp": new_timestamp
            })
    else:
        # new file not in baseline
        mismatches.append({
            "filename": new_filename,
            "status": "NEW",
            "old_hash": "N/A",
            "new_hash": new_hash,
            "timestamp": new_timestamp
        })

# check for deleted files
current_filenames = {filename for filename, _, _ in new_hashes}
for old_filename in previous_hashes:
    if old_filename not in current_filenames:
        mismatches.append({
            "filename": old_filename,
            "status": "DELETED",
            "old_hash": previous_hashes[old_filename],
            "new_hash": "N/A",
            "timestamp": datetime.now().isoformat()
        })

# display results
if mismatches:
    print("FILE INTEGRITY VIOLATIONS DETECTED:\n")
    for match in mismatches:
        print(f"File: {match['filename']}")
        print(f"Status: {match['status']}")
        print(f"Old Hash: {match['old_hash']}")
        print(f"New Hash: {match['new_hash']}")
        print(f"Timestamp: {match['timestamp']}")
        print("-" * 60)
else:
    print("All files match the baseline. No integrity violations detected.")


All files match the baseline. No integrity violations detected.


Change detection is a useful technique antivirus scanning software uses to catch suspicious file modifications that signature-based scans might miss, such as newly created, altered, or deleted files, even if they don't match known malware signatures. It also provides behavioural and forensic insight, helping trace intrusion timelines and affected files. 

Rootkits that operate at the kernel level can hide file access or intercept file system calls, can avoid change detection algorithms. The system might report no changes. 

Below is an extension of the system above that creates backups of clean files and automatically restores them when modifications or deletions are detected.

In [12]:
import os, hashlib, csv, shutil
from datetime import datetime

folder_path = './files'
backup_folder = './files_backup'
csv_output = "file_hashes.csv"

# Check if folder exists
if not os.path.exists(folder_path):
    print(f"❌ ERROR: Folder '{folder_path}' does not exist!")
    print(f"Current working directory: {os.getcwd()}")
    print(f"Creating folder...")
    os.makedirs(folder_path, exist_ok=True)
else:
    print(f"✅ Found folder: {os.path.abspath(folder_path)}")

# ensure backup folder exists
os.makedirs(backup_folder, exist_ok=True)

new_hashes = []
backup_count = 0

# loop through files in the folder
files_in_folder = os.listdir(folder_path)
print(f"Files found: {files_in_folder}\n")

for filename in files_in_folder:
    file_path = os.path.join(folder_path, filename)

    if os.path.isfile(file_path):
        # create backup of file
        backup_path = os.path.join(backup_folder, filename)
        shutil.copy2(file_path, backup_path)
        backup_count += 1
        print(f"✅ Backed up: {filename}")
        
        # compute hash
        hash = hashlib.sha256()
        with open(file_path, "rb") as f:
            # hash file in chunks of 4096 bytes
            for block in iter(lambda: f.read(4096), b""):
                hash.update(block)
        
        hash_hex = hash.hexdigest()
        timestamp = datetime.now().isoformat()

        new_hashes.append((filename, hash_hex, timestamp))


# load previous hashes from CSV into a dictionary
previous_hashes = {}

try:
    with open(csv_output, "r", newline="") as csvf:
        reader = csv.DictReader(csvf)
        for row in reader:
            previous_hashes[row["Filename"]] = row["SHA256 Hash"]
except FileNotFoundError:
    print("No baseline found. Cannot perform restoration.")

# compare new hashes with previous hashes
mismatches = []
for new_filename, new_hash, new_timestamp in new_hashes:
    if new_filename in previous_hashes:
        # file exists in baseline - check if hash matches
        if previous_hashes[new_filename] != new_hash:
            mismatches.append({
                "filename": new_filename,
                "status": "MODIFIED",
                "old_hash": previous_hashes[new_filename],
                "new_hash": new_hash,
                "timestamp": new_timestamp
            })
    else:
        # new file not in baseline
        mismatches.append({
            "filename": new_filename,
            "status": "NEW",
            "old_hash": "N/A",
            "new_hash": new_hash,
            "timestamp": new_timestamp
        })

# check for deleted files
current_filenames = {filename for filename, _, _ in new_hashes}
for old_filename in previous_hashes:
    if old_filename not in current_filenames:
        mismatches.append({
            "filename": old_filename,
            "status": "DELETED",
            "old_hash": previous_hashes[old_filename],
            "new_hash": "N/A",
            "timestamp": datetime.now().isoformat()
        })

# display results and restore files
if mismatches:
    print("\nFILE INTEGRITY VIOLATIONS DETECTED:\n")
    
    restored_count = 0
    failed_count = 0
    
    for match in mismatches:
        print(f"File: {match['filename']}")
        print(f"Status: {match['status']}")
        print(f"Old Hash: {match['old_hash']}")
        print(f"New Hash: {match['new_hash']}")
        print(f"Timestamp: {match['timestamp']}")
        
        # attempt restoration for MODIFIED and DELETED files
        if match['status'] in ['MODIFIED', 'DELETED']:
            backup_path = os.path.join(backup_folder, match['filename'])
            target_path = os.path.join(folder_path, match['filename'])
            
            if os.path.exists(backup_path):
                try:
                    shutil.copy2(backup_path, target_path)
                    print(f"RESTORED from backup")
                    restored_count += 1
                except Exception as e:
                    print(f"RESTORATION FAILED: {e}")
                    failed_count += 1
            else:
                print(f"No backup found - cannot restore")
                failed_count += 1
        elif match['status'] == 'NEW':
            print(f"New file detected - no restoration needed")
        
        print("-" * 60)
    
    print(f"\nRestoration Summary:")
    print(f"   Files restored: {restored_count}")
    print(f"   Restoration failures: {failed_count}")
    print(f"   Total violations: {len(mismatches)}")
else:
    print("\nAll files match the baseline. No integrity violations detected.")

print(f"\n{'=' * 60}")
print(f"Backup Summary:")
print(f"   Total files backed up: {backup_count}")
print(f"   Backup location: {os.path.abspath(backup_folder)}")
print(f"   Backup folder contents: {os.listdir(backup_folder)}")


✅ Found folder: c:\Users\NightFlash96\Desktop\Code\Y3 S1\Networks-and-System-Security-ePortfolio\Week4\files
Files found: ['kayak.txt', 'meower_supply.png', 'pangram.txt']

✅ Backed up: kayak.txt
✅ Backed up: meower_supply.png
✅ Backed up: pangram.txt

FILE INTEGRITY VIOLATIONS DETECTED:

File: kayak.txt
Status: MODIFIED
Old Hash: 2504bb7b2cd7e238e53aa25525d62f816039570ae6d8ed1a938861a63be8e532
New Hash: ef618843350f228fc9ee65c709b8f1829ff4d57bed46a4c3ae6c4f660b1eee28
Timestamp: 2025-12-17T06:59:25.743942
RESTORED from backup
------------------------------------------------------------
File: technician.png
Status: DELETED
Old Hash: d232d3ba5c26ec0ed4cc47c0476ab850de3e989ec60b99ee6f80859e291257f3
New Hash: N/A
Timestamp: 2025-12-17T06:59:25.746699
RESTORED from backup
------------------------------------------------------------

Restoration Summary:
   Files restored: 2
   Restoration failures: 0
   Total violations: 2

Backup Summary:
   Total files backed up: 3
   Backup location: c:\

As you can see both the modified and deleted file were restored after the code was run.

# Section 3: Signature-based Malware Detection

Key Concepts

- Signatures: Unique byte or code patterns associated with known malware.
- Pattern Matching: Scans files for predefined suspicious expressions or function calls.
- Weakness: Fails against obfuscated, encrypted, or polymorphic malware.
- Modern Shift: Behavioural analysis and machine learning now complement signature detection.

The code below loops through each file in the /files folder, and searches each file for these signatures. I created a demo malicious python file for this demo, since this code wouldn't work for image files such as PNGs. 

In [20]:
import re, os

target_path = './files'

# suspicious code signatures we are looking for
SIGNATURES = [
    r"eval\(",
    r"exec\(",
    r"base64\.b64decode",
    r"socket\.connect",
    r"import\s+os"
]

# compile regex patterns
compiled_signatures = [re.compile(sig) for sig in SIGNATURES]

def scan_file(file_path):
    findings = []
    line_number = 1

    #  use try in case file cannot be opened
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            for line in f:
                for pattern in compiled_signatures:
                    if pattern.search(line):
                        findings.append({
                            "file": file_path,
                            "line": line_number,
                            "pattern": pattern.pattern,
                            "code": line.strip()
                        })
                line_number += 1
    except Exception as e:
        print(f"Could not scan file {file_path}: {e}")

    return findings


def scan_directory(directory):
    all_findings = []

    for root, dirs, files in os.walk(directory):
        for filename in files: 
            if filename.endswith((".py", ".txt", ".png")): # include our txt and png files for the purpose of this demo
                file_path = os.path.join(root, filename)
                all_findings.extend(scan_file(file_path)) # scan each file if it matches
    return all_findings


results = scan_directory(target_path)

if not results:
    print("No malicious code signatures found.")
else:
    print("Malicious signatures detected:\n")
    for finding in results:
        print(f"File: {finding['file']}, Line: {finding['line']}")
        print(f"Pattern: {finding['pattern']}")
        print(f"Code: {finding['code']}\n")


Could not scan file ./files\meower_supply.png: 'utf-8' codec can't decode byte 0x89 in position 0: invalid start byte
Could not scan file ./files\technician.png: 'utf-8' codec can't decode byte 0x89 in position 0: invalid start byte
Malicious signatures detected:

File: ./files\demo.py, Line: 4
Pattern: import\s+os
Code: import os

File: ./files\demo.py, Line: 8
Pattern: eval\(
Code: def unsafe_eval(user_input):

File: ./files\demo.py, Line: 10
Pattern: eval\(
Code: eval(user_input)

File: ./files\demo.py, Line: 12
Pattern: exec\(
Code: def unsafe_exec(code):

File: ./files\demo.py, Line: 14
Pattern: exec\(
Code: exec(code)

File: ./files\demo.py, Line: 18
Pattern: base64\.b64decode
Code: return base64.b64decode(data)

File: ./files\demo.py, Line: 21
Pattern: socket\.connect
Code: # Suspicious: socket.connect

File: ./files\demo.py, Line: 26
Pattern: eval\(
Code: unsafe_eval("2 + 2")



The code successfully detects and lists all the signatures, correctly flagging the threat. However it is unable to scan PNG and TXT files (althought in theory, I thought TXT files would work too). For this code to work on PNGs, I would need to implement binary scanning, which could essentially be its own function, so for the purpose of this demo, I decided not to do that. Next time, I would definitely implement this feature though.

# Section 4: Worm Propagation Simulation

To visualize how a worm spreads rapidly across a network using random and opportunistic scanning strategies.

Key Concepts

- Worms: Self-contained programs that propagate through networks
without user action.
- Scanning: Worms randomly or strategically locate vulnerable
systems.
- Propagation Dynamics: The infection rate depends on network
topology and available hosts.
- Containment: Rate limiting, ingress filtering, and anomaly
detection reduce spread.

I decided to represent this worm network propagation simulation using an adjacency graph.

In [38]:
import random

# network graph (adjacency list)
network = {
    'A': ['B', 'C'],
    'B': ['A', 'D', 'E'],
    'C': ['A', 'F'],
    'D': ['B'],
    'E': ['B', 'F'],
    'F': ['C', 'E'],
    'G': ['C', 'F']
}

# Initial states
infected = set(['A'])  # initially infected node
infection_probability = 0.3  # probability of infection spread per connection

def propagate(infected, network):
    new_infected = set(infected)

    for node in infected:
        for neighbour in network[node]:
            if neighbour not in infected:
                if random.random() < infection_probability:
                    new_infected.add(neighbour)
    return new_infected

steps = 15 # number of propagation steps
for step in range(steps):
    print(f"Step {step}: Infected nodes: {infected}")
    infected = propagate(infected, network)

print(f"Final infected nodes after {steps} steps: {infected}")

Step 0: Infected nodes: {'A'}
Step 1: Infected nodes: {'A'}
Step 2: Infected nodes: {'A', 'C'}
Step 3: Infected nodes: {'A', 'C'}
Step 4: Infected nodes: {'A', 'C'}
Step 5: Infected nodes: {'A', 'C'}
Step 6: Infected nodes: {'A', 'B', 'C'}
Step 7: Infected nodes: {'E', 'A', 'B', 'C'}
Step 8: Infected nodes: {'E', 'A', 'F', 'B', 'C'}
Step 9: Infected nodes: {'C', 'A', 'F', 'E', 'B'}
Step 10: Infected nodes: {'C', 'A', 'F', 'E', 'B'}
Step 11: Infected nodes: {'C', 'D', 'A', 'F', 'E', 'B'}
Step 12: Infected nodes: {'C', 'D', 'A', 'F', 'E', 'B'}
Step 13: Infected nodes: {'C', 'D', 'A', 'F', 'E', 'B'}
Step 14: Infected nodes: {'C', 'D', 'A', 'F', 'E', 'B'}
Final infected nodes after 15 steps: {'C', 'D', 'A', 'F', 'E', 'B'}


As you can see from the nodes being printed, there is a clearly visible "infection curve".

Doubling infection attempts per host would make the infection curve steeper by a factor of n^2. The worm would spread exponentially quicker.

Worms will often choose a local subnet as a target for propagation, as machines nearby on the same OS, with fewer firewalls between them will result in a higher infection success rate. Faster early spread will also increase spreading rate later on in the propagation process.

The most effective containment strategy for a worm in this kind of network would be **scan detection** because it can identify rapid propagation. Rate halting and thresholding help as countermeasures, but scan detection is better as a proactive method.

# Section 5: Countermeasure Design Challenge

**Key Concepts**

- Defence in Depth: Layering multiple, complementary protection methods.
- Host-based vs. Network-based: Combining internal integrity monitoring with perimeter defences.
- User Awareness: Social engineering remains one of the weakest links.
- Resilience: Recovery and containment strategies matter as much as prevention.

**Task**

Students form small groups and design a Python-based monitoring prototype that:

- Detects unusual network activity (e.g., excessive outbound connections).
- Logs or alerts administrators when anomalies are found.
- Uses existing code from previous exercises as building blocks.

In [2]:
import os, csv, subprocess, time
from collections import Counter
from datetime import datetime

# This prototype mirrors earlier baseline-vs-current comparisons: we build a trusted view
# of normal connection volume and host spread, then alert on deltas that exceed thresholds.
baseline_file = "net_activity_baseline.csv"
alert_log = "network_alerts.log"

thresholds = {
    "max_total": 60,        # hard cap on total outbound connections
    "max_per_host": 12,     # too many sockets to the same host
    "max_new_hosts": 8      # how many more distinct hosts than baseline
}

monitor_interval = 3  # seconds between samples
sample_count = 3       # how many samples to take in this demo run


def get_outbound_connections():
    """Parse netstat output to collect outbound TCP connections with their remote hosts."""
    # On Windows, netstat is available by default; using subprocess keeps dependencies minimal.
    result = subprocess.run(["netstat", "-ano"], capture_output=True, text=True, check=False)
    connections = []

    # loop through output lines
    for line in result.stdout.splitlines():
        line = line.strip()
        if not line.startswith("TCP"): # only care about TCP lines
            continue

        # if not enough parts, skip
        parts = line.split()
        if len(parts) < 4: 
            continue

        _, local_ep, remote_ep, state, *rest = parts
        # keep established/active-ish states only
        if state.upper() not in {"ESTABLISHED", "SYN_SENT", "SYN_SEND", "TIME_WAIT"}:
            continue

        # if either endpoint lacks port, skip
        if ":" not in local_ep or ":" not in remote_ep:
            continue

        # extract IPs only
        local_ip, _ = local_ep.rsplit(":", 1)
        remote_ip, _ = remote_ep.rsplit(":", 1)

        # ignore loopback and unroutable placeholders
        if remote_ip.startswith("127.") or remote_ip == "0.0.0.0":
            continue
        if local_ip.startswith("127."):
            continue

        connections.append({"local": local_ip, "remote": remote_ip, "state": state.upper()})

    return connections


def compute_metrics(connections):
    """Summarise connection volume and host spread."""
    per_host = Counter(conn["remote"] for conn in connections) # count per remote host
    return {
        "total": len(connections),
        "hosts": len(per_host),
        "per_host": per_host,
    }


def load_baseline():
    if not os.path.exists(baseline_file):
        return None
    with open(baseline_file, "r", newline="") as f:
        reader = csv.DictReader(f)
        row = next(reader, None)
        if not row:
            return None
        return {
            "avg_total": int(row["avg_total"]),
            "avg_hosts": int(row["avg_hosts"])
        }


def write_baseline(metrics):
    with open(baseline_file, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["avg_total", "avg_hosts"])
        writer.writeheader()
        writer.writerow({
            "avg_total": metrics["total"],
            "avg_hosts": metrics["hosts"]
        })


def detect_anomalies(metrics, baseline):
    reasons = []

    if metrics["total"] > thresholds["max_total"]:
        reasons.append(f"total outbound connections {metrics['total']} exceeds {thresholds['max_total']}")

    if metrics["hosts"] - baseline["avg_hosts"] > thresholds["max_new_hosts"]:
        reasons.append(
            f"distinct remote hosts jumped by {metrics['hosts'] - baseline['avg_hosts']} (limit {thresholds['max_new_hosts']})"
        )

    hot_hosts = [host for host, count in metrics["per_host"].items() if count > thresholds["max_per_host"]]
    for host in hot_hosts:
        reasons.append(f"host {host} has {metrics['per_host'][host]} connections (limit {thresholds['max_per_host']})")

    return reasons


def log_alert(reasons, metrics):
    ts = datetime.now().isoformat()
    lines = [
        f"[{ts}] ALERT: {'; '.join(reasons)}",
        f"    total={metrics['total']} hosts={metrics['hosts']} top_hosts={metrics['per_host'].most_common(3)}"
    ]

    with open(alert_log, "a", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")

    print("\n".join(lines))


# ---- main demo run ----
print("Bootstrapping baseline if missing and sampling network activity...\n")
baseline = load_baseline()

if baseline is None:
    first_sample = compute_metrics(get_outbound_connections())
    write_baseline(first_sample)
    baseline = load_baseline()
    print(f"Created baseline from current activity: avg_total={baseline['avg_total']} avg_hosts={baseline['avg_hosts']}")

for i in range(sample_count):
    connections = get_outbound_connections()
    metrics = compute_metrics(connections)
    reasons = detect_anomalies(metrics, baseline)

    print(f"Sample {i+1}/{sample_count}: total={metrics['total']} hosts={metrics['hosts']}")

    if reasons:
        log_alert(reasons, metrics)
    else:
        print("No anomalies relative to baseline/thresholds.")

    if i < sample_count - 1:
        time.sleep(monitor_interval)

print("\nFinished monitoring window. Check network_alerts.log for any alerts written.")

Bootstrapping baseline if missing and sampling network activity...

Sample 1/3: total=31 hosts=21
No anomalies relative to baseline/thresholds.
Sample 2/3: total=33 hosts=21
[2025-12-17T11:12:38.047677] ALERT: host 192.168.1.1 has 13 connections (limit 12)
    total=33 hosts=21 top_hosts=[('192.168.1.1', 13), ('34.141.58.12', 1), ('40.79.150.122', 1)]
Sample 3/3: total=37 hosts=22
[2025-12-17T11:12:41.061185] ALERT: host 192.168.1.1 has 15 connections (limit 12)
    total=37 hosts=22 top_hosts=[('192.168.1.1', 15), ('[2a02:6b67:d210:8e00:daec:5eff:fe86:7699]', 2), ('34.141.58.12', 1)]

Finished monitoring window. Check network_alerts.log for any alerts written.


The code above detected three anomalies: 
1. Host 192.168.1.1 suddenly spikes to 13 connections (exceeds the 12-connection threshold).
2. Same host escalates to 15 connections—the threat is worsening.
3. A new IPv6 host also appears.

This pattern could be characteristic of malware behaviour. It could indicate a scanning/propagation attempt, where a single host is making repeated connections to an external IP in order to propagate a worm. The jump from 13 to 15 connections could suggest the malware is increasing its activity, which could not be just coincidental traffic spikes. 

The system correctly:
- Built a bassline
- Detected threshold violations
- Logged timestamps and details for forensics analysis
- Identified the specific problem host so that it can be quarantined/isolated

The issue:
- The alerts don't show context. Real systems would correlate with process monitoring. For all we know, all of this could just be a false alarm.
- The system doesn't have a built in automatic response. No automatic blocking, alerting admins, quarantining hosts, etc. That would be the next step to take.

### Discussion points:

**First layer of faliure** in most real-world malware outbreaks is the user layer. People click malicious links in emails, open infected attachments or use weak passwords. Technical defenses like firewalls and antivirus programs often act as incident-response, after the user makes a mistake.

**AI-based detection systems** can be trained on large, labelled datasets of normal system behaviour and known malicious activity. **Anomaly detection** can be used by AI to spot patterns that deviate from the norm, even if the exact malware hasn't been seen before.

**Balancing sensitivity and false positives** requires security teams to fine-tune detection thresholds. A risk-based approach is often used. Focusing more attention on critical assets like servers holding sensitive data. Alerts are also contextualised, for example and unusual login on a financial system is more critical than on a public website. Detection accuracy is upheld by continuous review and tuning, using historical data and feedback loops.